# 2. Model Training

This notebook trains the 5-tier HP prediction models using engineered features.

**Input:** `helper_files/engineered_features.parquet`  
**Output:** `pickled_models/hp_model_cr*.pkl`

## Imports and Configs

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path
from sklearn.preprocessing import StandardScaler



📁 Execution context detected:
   Current directory: /workspaces/matrix_v0
   Data directory: ./data
   Models directory: ./pickled_models
Imports successful


In [ ]:
# Detect execution context and set paths dynamically
sys.path.insert(0, '.')

# Get the current working directory
cwd = Path.cwd()

# Check if we're in the notebooks directory or project root
if cwd.name == 'notebooks':
    # Running from notebooks directory (in Jupyter)
    DATA_DIR = '../data'
    PICKLED_MODELS_DIR = '../pickled_models'
    MONSTER_BUILDER_DIR = '../monster-builder-v2'
    HELPERS_DIR = './helper_files'
    IN_NB_DIR = False
else:
    # Running from project root (via run_three_tier_model.py)
    DATA_DIR = './data'
    PICKLED_MODELS_DIR = './pickled_models'
    MONSTER_BUILDER_DIR = './monster-builder-v2'
    HELPERS_DIR = './notebooks/helper_files'
    IN_NB_DIR = False

print(f"📁 Execution context detected:")
print(f"   Current directory: {cwd}")
print(f"   Data directory: {DATA_DIR}")
print(f"   Models directory: {PICKLED_MODELS_DIR}")

print("Imports successful")

In [ ]:
# Add helper_files to path
if IN_NB_DIR is True:
    from helper_files import (
        get_phase3_features,
        train_constrained_model,
        ConstrainedModel,
        calculate_r2,
        calculate_mae,
        save_model,
        summarize_model_performance,
        extract_family,
    )

    print("Imports successful")
else:
    from notebooks.helper_files import (
        get_phase3_features,
        train_constrained_model,
        ConstrainedModel,
        calculate_r2,
        calculate_mae,
        save_model,
        summarize_model_performance,
        extract_family,
    )  

    print("Imports successful")


Imports successful


## Load Engineered Features

In [3]:
# Load engineered features
load_path = HELPERS_DIR + "/engineered_features.parquet"
df = pd.read_parquet(load_path)
print(f"Loaded {len(df)} monsters with {len(df.columns)} features")

Loaded 382 monsters with 129 features


In [4]:
# Get Phase 3 features
phase3_features = get_phase3_features()
print(f"Phase 3 features: {len(phase3_features)}")

Phase 3 features: 38


## Split by CR Tier

In [5]:
# Split by CR tier
df_cr1 = df[df['cr_tier'] == 'cr1'].copy()
df_cr2 = df[df['cr_tier'] == 'cr2'].copy()
df_cr3 = df[df['cr_tier'] == 'cr3'].copy()
df_cr4 = df[df['cr_tier'] == 'cr4'].copy()
df_cr5 = df[df['cr_tier'] == 'cr5'].copy()

print(f"CR < 1:    {len(df_cr1)} monsters")
print(f"CR 1-4:    {len(df_cr2)} monsters")
print(f"CR 5-10:   {len(df_cr3)} monsters")
print(f"CR 11-16:  {len(df_cr4)} monsters")
print(f"CR > 16:   {len(df_cr5)} monsters")

CR < 1:    142 monsters
CR 1-4:    124 monsters
CR 5-10:   67 monsters
CR 11-16:  28 monsters
CR > 16:   21 monsters


## Train/Test Split Strategy

Using family-based splitting to avoid data leakage.

In [6]:
# Manual train/test split by creature family
# Simple creatures -> all training
# Complex creatures -> split evenly

def manual_train_test_split(df_tier, test_ratio=0.2, random_state=42):
    """Split data using family-based strategy."""
    np.random.seed(random_state)
    
    # Simple families (all training)
    simple_families = ['beast', 'humanoid', 'giant']
    
    # Creatures from simple families
    simple_mask = df_tier['family'].isin(simple_families)
    
    # Get unique complex families
    complex_families = df_tier[~simple_mask]['family'].unique()
    
    # Shuffle and split complex families
    np.random.shuffle(complex_families)
    n_test = max(1, int(len(complex_families) * test_ratio))
    test_families = set(complex_families[:n_test])
    
    # Create masks
    train_mask = simple_mask | ~df_tier['family'].isin(test_families)
    test_mask = ~train_mask
    
    return df_tier[train_mask], df_tier[test_mask]

# For small tiers (CR 11-16, CR > 16), use all data for training
def split_by_tier(df_tier, tier_name):
    if len(df_tier) < 30:
        print(f"  {tier_name}: Using all {len(df_tier)} samples for training (small tier)")
        return df_tier, df_tier  # Train and test on same data for small tiers
    else:
        train, test = manual_train_test_split(df_tier)
        print(f"  {tier_name}: {len(train)} train, {len(test)} test")
        return train, test

In [7]:
# Split each tier
print("Splitting data:")
train_cr1, test_cr1 = split_by_tier(df_cr1, 'CR < 1')
train_cr2, test_cr2 = split_by_tier(df_cr2, 'CR 1-4')
train_cr3, test_cr3 = split_by_tier(df_cr3, 'CR 5-10')
train_cr4, test_cr4 = split_by_tier(df_cr4, 'CR 11-16')
train_cr5, test_cr5 = split_by_tier(df_cr5, 'CR > 16')

Splitting data:
  CR < 1: 125 train, 17 test
  CR 1-4: 100 train, 24 test
  CR 5-10: 59 train, 8 test
  CR 11-16: Using all 28 samples for training (small tier)
  CR > 16: Using all 21 samples for training (small tier)


## Train Models

In [8]:
def train_tier_model(train_df, test_df, tier_name, phase3_features):
    """Train a model for a single CR tier."""
    print(f"\n{'='*60}")
    print(f"Training {tier_name} model...")
    print(f"{'='*60}")
    
    # Prepare features
    X_train = train_df[phase3_features].fillna(0).values
    y_train = train_df['residual_hp'].values
    
    X_test = test_df[phase3_features].fillna(0).values
    y_test = test_df['residual_hp'].values
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train constrained model
    coefficients, intercept = train_constrained_model(
        X_train_scaled, y_train, phase3_features, scaler
    )
    
    # Create model object
    model = ConstrainedModel(coefficients, intercept)
    
    # Evaluate on test set
    y_pred_residual = model.predict(X_test_scaled)
    
    # Calculate full HP predictions
    y_pred_hp = test_df['hp_after_phase2'].values + y_pred_residual
    y_actual_hp = test_df['actual_hp'].values
    
    # Calculate metrics
    r2 = calculate_r2(y_actual_hp, y_pred_hp)
    mae = calculate_mae(y_actual_hp, y_pred_hp)
    
    print(f"\n{tier_name} Results:")
    print(f"   Training samples: {len(y_train)}")
    print(f"   Test R²:  {r2:.4f}")
    print(f"   Test MAE: {mae:.2f} HP")
    
    return {
        'model': model,
        'scaler': scaler,
        'train_count': len(y_train),
        'test_r2': r2,
        'test_mae': mae,
    }

In [9]:
# Train all models
results = {}

results['cr1'] = train_tier_model(train_cr1, test_cr1, 'CR < 1', phase3_features)
results['cr2'] = train_tier_model(train_cr2, test_cr2, 'CR 1-4', phase3_features)
results['cr3'] = train_tier_model(train_cr3, test_cr3, 'CR 5-10', phase3_features)
results['cr4'] = train_tier_model(train_cr4, test_cr4, 'CR 11-16', phase3_features)
results['cr5'] = train_tier_model(train_cr5, test_cr5, 'CR > 16', phase3_features)


Training CR < 1 model...

CR < 1 Results:
   Training samples: 125
   Test R²:  -16.1651
   Test MAE: 31.84 HP

Training CR 1-4 model...

CR 1-4 Results:
   Training samples: 100
   Test R²:  -2.5528
   Test MAE: 36.35 HP

Training CR 5-10 model...

CR 5-10 Results:
   Training samples: 59
   Test R²:  0.1533
   Test MAE: 31.62 HP

Training CR 11-16 model...

CR 11-16 Results:
   Training samples: 28
   Test R²:  -0.5386
   Test MAE: 49.40 HP

Training CR > 16 model...

CR > 16 Results:
   Training samples: 21
   Test R²:  0.9332
   Test MAE: 29.81 HP


In [10]:
# Summary
summarize_model_performance(results)


5-BUCKET HP MODEL TRAINING COMPLETE

MODEL PERFORMANCE SUMMARY:

   CR < 1 Model:
      Training samples: 125
      Test R²:  -16.1651
      Test MAE: 31.84 HP

   CR 1-4 Model:
      Training samples: 100
      Test R²:  -2.5528
      Test MAE: 36.35 HP

   CR 5-10 Model:
      Training samples: 59
      Test R²:  0.1533
      Test MAE: 31.62 HP

   CR 11-16 Model:
      Training samples: 28
      Test R²:  -0.5386
      Test MAE: 49.40 HP

   CR > 16 Model:
      Training samples: 21
      Test R²:  0.9332
      Test MAE: 29.81 HP

All 5 models trained successfully!


## Save Models

'./pickled_models'

In [17]:


# Ensure output directory exists
os.makedirs(PICKLED_MODELS_DIR, exist_ok=True)
save_path = PICKLED_MODELS_DIR + "/hp_model_tier.pkl"
# Save each model
for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']:

    filepath = save_path.replace('tier', tier)
    save_model(
        results[tier]['model'],
        results[tier]['scaler'],
        phase3_features,
        filepath
    )
    print(f"Saved {tier} model to {filepath}")

print("\nAll models saved successfully!")

Saved cr1 model to ./pickled_models/hp_model_cr1.pkl
Saved cr2 model to ./pickled_models/hp_model_cr2.pkl
Saved cr3 model to ./pickled_models/hp_model_cr3.pkl
Saved cr4 model to ./pickled_models/hp_model_cr4.pkl
Saved cr5 model to ./pickled_models/hp_model_cr5.pkl

All models saved successfully!


In [18]:
# Display top feature coefficients for each tier
for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']:
    print(f"\n{tier.upper()} Top Features by Coefficient:")
    coefs = results[tier]['model'].coef_
    coef_df = pd.DataFrame({
        'feature': phase3_features,
        'coefficient': coefs
    }).sort_values('coefficient', key=abs, ascending=False)
    
    print(coef_df.head(10).to_string(index=False))


CR1 Top Features by Coefficient:
                feature  coefficient
            trait_count     8.111764
   movement_types_count     7.244964
           speed_burrow    -5.342879
 speed_ground_deviation     5.243194
 inflicts_incapacitated    -4.251160
            speed_climb    -4.191616
    total_ability_count    -3.779103
skill_proficiency_count    -2.772423
             speed_swim    -2.558495
      inflicts_deafened    -2.546089

CR2 Top Features by Coefficient:
               feature  coefficient
        reaction_count     9.993559
           speed_climb    -9.574806
   speed_fly_deviation    -8.545917
size_ordinal_deviation     6.723918
           trait_count     6.561994
save_proficiency_count    -6.348258
  darkvision_deviation     5.388128
            speed_swim    -5.018031
   vulnerability_count     4.997891
  movement_types_count     4.991102

CR3 Top Features by Coefficient:
                    feature  coefficient
        total_ability_count    21.127980
    skill_pro